In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

sa = "stretaillakeatharva01"
silver = f"abfss://silver@{sa}.dfs.core.windows.net"
dim_path = f"{silver}/dim_customers"

TRACKED = ["customer_fname", "customer_lname", "address", "city", "state", "pincode"]

def add_hash(df):
    return df.withColumn(
        "row_hash",
        F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in TRACKED]), 256))

In [0]:
cust = spark.read.format("delta").load(f"{silver}/customers")

dim = (add_hash(cust)
    .withColumn("effective_start", F.lit("2013-01-01").cast("date"))
    .withColumn("effective_end", F.lit("9999-12-31").cast("date"))
    .withColumn("is_current", F.lit(True)))

dim.write.format("delta").mode("overwrite").save(dim_path)
print("dim rows:", spark.read.format("delta").load(dim_path).count())

dim rows: 12435


In [0]:
base = spark.read.format("delta").load(f"{silver}/customers")

# 5 customers ka address/city badla
changed = (base.filter(F.col("customer_id").isin(1, 2, 3, 4, 5))
    .withColumn("address", F.lit("100 New Street"))
    .withColumn("city", F.lit("Pune"))
    .withColumn("state", F.lit("MH"))
    .withColumn("pincode", F.lit(411001)))

# 1 bilkul naya customer
new_cust = (base.limit(1)
    .withColumn("customer_id", F.lit(99999))
    .withColumn("customer_fname", F.lit("Atharva"))
    .withColumn("customer_lname", F.lit("Test")))

# baaki sab unchanged
unchanged = base.filter(~F.col("customer_id").isin(1, 2, 3, 4, 5))

incoming = add_hash(changed.unionByName(new_cust).unionByName(unchanged))
print("incoming rows:", incoming.count())

incoming rows: 12436


In [0]:
dim_tbl = DeltaTable.forPath(spark, dim_path)
current = dim_tbl.toDF().filter("is_current = true").select("customer_id", F.col("row_hash").alias("old_hash"))

# Jin rows ka hash badla, unki NAYI version insert karni hai (merge_key = null)
changed_rows = (incoming.join(current, "customer_id")
    .filter(F.col("row_hash") != F.col("old_hash"))
    .drop("old_hash")
    .withColumn("merge_key", F.lit(None).cast("int")))

# Sab incoming rows: match karne ke liye merge_key = customer_id
all_rows = incoming.withColumn("merge_key", F.col("customer_id"))

staged = changed_rows.unionByName(all_rows)

(dim_tbl.alias("t")
  .merge(staged.alias("s"), "t.customer_id = s.merge_key AND t.is_current = true")
  .whenMatchedUpdate(
      condition="t.row_hash <> s.row_hash",
      set={"is_current": "false", "effective_end": "current_date()"})
  .whenNotMatchedInsert(values={
      **{c: f"s.{c}" for c in ["customer_id", "customer_fname", "customer_lname", "username",
                                "address", "city", "state", "pincode", "silver_load_ts", "row_hash"]},
      "effective_start": "current_date()",
      "effective_end": "cast('9999-12-31' as date)",
      "is_current": "true"})
  .execute())

d = spark.read.format("delta").load(dim_path)
print("total rows:", d.count(), "| current rows:", d.filter("is_current").count())

total rows: 12441 | current rows: 12436


In [0]:
display(d.filter(F.col("customer_id").isin(1, 2, 99999))
         .select("customer_id", "city", "state", "effective_start", "effective_end", "is_current")
         .orderBy("customer_id", "effective_start"))

display(spark.sql(f"DESCRIBE HISTORY delta.`{dim_path}`").select("version", "timestamp", "operation"))

customer_id,city,state,effective_start,effective_end,is_current
1,Brownsville,TX,2013-01-01,2026-10-19,false
1,Pune,MH,2026-10-19,9999-12-31,true
2,Littleton,CO,2013-01-01,2026-10-19,false
2,Pune,MH,2026-10-19,9999-12-31,true
99999,San Antonio,TX,2026-10-19,9999-12-31,true


version,timestamp,operation
1,2026-10-19T16:36:23.000Z,MERGE
0,2026-10-19T16:36:00.000Z,WRITE
